In [741]:
import requests
import json

properties = requests.get('http://127.0.0.1:3000/api/properties')
properties_by_garment_key = {}

for p in properties.json():
    if p['garmentKey'] not in properties_by_garment_key:
        properties_by_garment_key[p['garmentKey']] = {}
    properties_by_garment_key[p['garmentKey']][p['garmentValue']] = p




In [742]:
import re

def process_garment(garment):
    type = garment['model_number'][0]
    gender = garment['model_number'][1]
    procedure = garment['model_number'].split('-')[1:] if '-' in garment['model_number'] else None
    material = 'X-NIGHT' if 'X-NIGHT' in garment['material'] else 'IN-BETWEEN' if 'IN-BETWEEN' in garment['material'] else 'IN-ORDER' if 'IN-ORDER' in garment['material'] else garment['material'].split(" ")[0].split('-')[0]
    process = None if 'X-NIGHT' in garment['material'] else None if 'IN-BETWEEN' in garment['material'] or 'IN-ORDER' in garment['material'] else garment['material'].split('-')[1].split(' ')[0] if len(garment['material'].split('-')) > 1 else None
    color_match = re.match(r'^([0-9*]+)', garment['colour']) if garment['colour'] else None
    color = color_match.group(1) if color_match else garment['colour']
    if color and len(color) == 1:
        color = '0' + color
    title = garment['product_name']
    model_match = re.search(r'/(\d+)', garment['model_number'])
    model = model_match.group(1) if model_match else garment['model_number']
    payload = {
        'category': None,
        'type': properties_by_garment_key['type'][type]['garmentValue'] if type else None,
        'gender': properties_by_garment_key['gender'][gender]['garmentValue'] if gender else None,
        'procedure': [properties_by_garment_key['procedure'][p]['garmentValue'] for p in procedure] if procedure else None,
        'material': properties_by_garment_key['material'][material]['garmentValue'] if material else None,
        'process': properties_by_garment_key['process'][process]['garmentValue'] if process else None,
        'color': properties_by_garment_key['color'][color]['garmentValue'] if color else None,
        'title': title,
        'model': model if model else None,
        'images': garment['images'],
        'uploadedByUserId': 'user_3AFaW6Rkhu0b2vF3JZMEDQDJOiv',
        'source': {
            "type": "external",
            "label": "The Library",
            "url": garment['url']
        }
    }
    return payload


In [743]:
failures = set()
successes = set()

In [744]:


from pathlib import Path

products_dir = Path("products")
out_dir = Path("processed_products")
import shutil

if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

for json_path in sorted(products_dir.glob("*.json")):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            garment = json.load(f)
        payload = process_garment(garment)
        out_path = out_dir / json_path.name
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        print(json_path.name)
    except Exception as e:
        print(f"Error processing {json_path.name}: {e}")
        failures.add(json_path.name)




AF-0874-ORG-36_c62c1b8e-903a-43e0-8377-bd9abaed2b30.json
AF-0900L-BNE-39_22a425e2-708b-4e04-9e72-4f2eeda21fa3.json
AF-0901L-36-39_07983b0a-a06d-46f3-800c-30db2b4cfba5.json
AF-0901L-36_6ab5cbee-f684-4011-bd95-3716cae3810c.json
AF-0901L-36_7cbb8623-ba5c-457c-9989-a438e28f1092.json
AF-0907-010-39_68303e55-c20d-45c8-92fe-28a0f0ae2e25.json
AF-0991L-01-39_f03148f8-0bbb-45d5-af8e-6c1d7aaeef69.json
AF-0994P-36-37_f21ce077-579f-4a73-ae97-8d741e7593b4.json
AM-2452-13_ee932535-1bd5-4f49-a6e7-e3a8d2040d16.json
AM-2598-IN-6_ece11699-cf09-45c6-8296-e07d44548964.json
AM-2689-IN-01-11 CCP-IN-57_a0a5e27e-2aad-44b6-8988-dcb054c824ed.json
AM-2689-IN-19-7_e7247019-270a-4b2e-bf60-18e85083d9af.json
AM-452-10_be9c9179-4499-4eae-80af-82e4fe7d715b.json
CC-IN-55-38_373fb8de-4b4d-412f-9c8a-fc03a4bd460a.json
CCP-IN-100W-40_89b56b66-6826-4256-907c-b632077526f5.json
CCP-IN-103-52_921067cd-1ee6-4b90-88cb-c705d065cc5f.json
CCP-IN-104-10_6237a1c3-f0ca-41cc-8ad8-caaa3a8bfc22.json
CCP-IN-105-11_b8d26c87-b27d-48a6-baaa-0

In [745]:
len(failures)


0

In [746]:
for filename in failures:
    with open('products/' + filename, 'r') as f:
        garment = json.load(f)
        print(garment['url'])
        print(garment['sku'])
    payload = process_garment(garment)
    out_path = out_dir / filename
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(filename)


